In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/processed/cleaned_shots.csv")

df.head()

,match_id,competition_name,season_name,minute,second,period,team,player,position,shot_body_part,...,shot_one_on_one,shot_open_goal,shot_deflected,shot_statsbomb_xg,shot_outcome,x,y,end_x,end_y,goal
0,3923881,African Cup of Nations,2023,3,36,1,Côte d'Ivoire,Sébastien Haller,Center Forward,Head,...,0,0,0,0.077357,Wayward,106.7,42.1,115.6,48.4,0
1,3923881,African Cup of Nations,2023,6,10,1,Côte d'Ivoire,Sébastien Haller,Center Forward,Right Foot,...,0,0,0,0.187051,Off T,109.1,35.5,120.0,46.6,0
2,3923881,African Cup of Nations,2023,7,8,1,Côte d'Ivoire,Simon Adingra,Left Wing,Right Foot,...,0,0,0,0.013922,Off T,113.6,23.8,120.0,45.7,0
3,3923881,African Cup of Nations,2023,13,4,1,Côte d'Ivoire,Seko Fofana,Left Center Midfield,Right Foot,...,0,0,0,0.027030,Saved,91.8,40.2,118.1,40.2,0
4,3923881,African Cup of Nations,2023,20,1,1,Côte d'Ivoire,Obite Evan Ndicka,Left Center Back,Head,...,0,0,0,0.030268,Wayward,111.0,31.3,113.6,34.4,0


In [3]:
def calculate_angle(x, y):
    goal_width = 8

    left_post_y = 40 - (goal_width / 2)
    right_post_y = 40 + (goal_width / 2)

    angle = np.arctan2(
        right_post_y - y,
        120 - x
    ) - np.arctan2(
        left_post_y - y,
        120 - x
    )

    angle = np.abs(angle)

    return angle

In [4]:
df["shot_angle"] = df.apply(
    lambda row: calculate_angle(row["x"], row["y"]),
    axis=1
)

In [5]:
df["distance_from_center"] = abs(
    40 - df["y"]
)

In [6]:
binary_columns = [
    "under_pressure",
    "shot_first_time",
    "shot_one_on_one",
    "shot_open_goal",
    "shot_deflected"
]

for col in binary_columns:
    df[col] = df[col].astype(int)

In [7]:
categorical_columns = [
    "shot_body_part",
    "shot_technique",
    "play_pattern"
]

df = pd.get_dummies(
    df,
    columns=categorical_columns,
    drop_first=True
)

In [8]:
df.head()

,match_id,competition_name,season_name,minute,second,period,team,player,position,under_pressure,...,shot_technique_Overhead Kick,shot_technique_Volley,play_pattern_From Counter,play_pattern_From Free Kick,play_pattern_From Goal Kick,play_pattern_From Keeper,play_pattern_From Kick Off,play_pattern_From Throw In,play_pattern_Other,play_pattern_Regular Play
0,3923881,African Cup of Nations,2023,3,36,1,Côte d'Ivoire,Sébastien Haller,Center Forward,1,...,False,False,False,True,False,False,False,False,False,False
1,3923881,African Cup of Nations,2023,6,10,1,Côte d'Ivoire,Sébastien Haller,Center Forward,0,...,False,False,False,False,False,False,False,False,False,False
2,3923881,African Cup of Nations,2023,7,8,1,Côte d'Ivoire,Simon Adingra,Left Wing,0,...,False,False,False,False,False,False,False,False,False,True
3,3923881,African Cup of Nations,2023,13,4,1,Côte d'Ivoire,Seko Fofana,Left Center Midfield,0,...,False,False,False,False,False,False,False,False,False,False
4,3923881,African Cup of Nations,2023,20,1,1,Côte d'Ivoire,Obite Evan Ndicka,Left Center Back,0,...,False,False,False,False,False,False,False,False,False,False


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7862 entries, 0 to 7861
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   match_id                      7862 non-null   int64  
 1   competition_name              7862 non-null   object 
 2   season_name                   7862 non-null   int64  
 3   minute                        7862 non-null   int64  
 4   second                        7862 non-null   int64  
 5   period                        7862 non-null   int64  
 6   team                          7862 non-null   object 
 7   player                        7862 non-null   object 
 8   position                      7862 non-null   object 
 9   under_pressure                7862 non-null   int64  
 10  shot_first_time               7862 non-null   int64  
 11  shot_one_on_one               7862 non-null   int64  
 12  shot_open_goal                7862 non-null   int64  
 13  sho

In [10]:
df["shot_distance"] = np.sqrt(
    (120 - df["x"])**2 +
    (40 - df["y"])**2
)

In [12]:
bool_columns = df.select_dtypes(include="bool").columns

df[bool_columns] = df[bool_columns].astype(int)

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7862 entries, 0 to 7861
Data columns (total 41 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   match_id                      7862 non-null   int64  
 1   competition_name              7862 non-null   object 
 2   season_name                   7862 non-null   int64  
 3   minute                        7862 non-null   int64  
 4   second                        7862 non-null   int64  
 5   period                        7862 non-null   int64  
 6   team                          7862 non-null   object 
 7   player                        7862 non-null   object 
 8   position                      7862 non-null   object 
 9   under_pressure                7862 non-null   int64  
 10  shot_first_time               7862 non-null   int64  
 11  shot_one_on_one               7862 non-null   int64  
 12  shot_open_goal                7862 non-null   int64  
 13  sho

In [14]:
columns_to_drop = [
    "match_id",
    "competition_name",
    "season_name",
    "team",
    "player",
    "position",
    "shot_outcome",
    "shot_statsbomb_xg"
]

df = df.drop(columns=columns_to_drop)

In [16]:
corr = df.corr(numeric_only=True)["goal"].sort_values(
    ascending=False
)

print(corr)

goal                            1.000000
play_pattern_Other              0.384379
shot_angle                      0.287912
end_x                           0.242424
period                          0.229908
x                               0.192804
shot_open_goal                  0.167654
minute                          0.150918
shot_one_on_one                 0.074112
shot_deflected                  0.066275
shot_technique_Lob              0.048458
shot_body_part_Right Foot       0.028001
shot_technique_Normal           0.023365
play_pattern_From Counter       0.021361
shot_first_time                 0.020562
shot_technique_Diving Header    0.009337
second                          0.006821
y                               0.005208
play_pattern_From Keeper        0.004413
shot_body_part_Other           -0.001780
end_y                          -0.004755
shot_technique_Volley          -0.005974
shot_body_part_Left Foot       -0.011071
play_pattern_From Goal Kick    -0.015768
play_pattern_Fro

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7862 entries, 0 to 7861
Data columns (total 33 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   minute                        7862 non-null   int64  
 1   second                        7862 non-null   int64  
 2   period                        7862 non-null   int64  
 3   under_pressure                7862 non-null   int64  
 4   shot_first_time               7862 non-null   int64  
 5   shot_one_on_one               7862 non-null   int64  
 6   shot_open_goal                7862 non-null   int64  
 7   shot_deflected                7862 non-null   int64  
 8   x                             7862 non-null   float64
 9   y                             7862 non-null   float64
 10  end_x                         7862 non-null   float64
 11  end_y                         7862 non-null   float64
 12  goal                          7862 non-null   int64  
 13  sho

In [17]:
df.to_csv(
    "../data/processed/feature_engineered_shots.csv",
    index=False
)

print("Feature engineered dataset saved successfully.")

Feature engineered dataset saved successfully.
